# L'étage de la voix : affiner Soloni sur vos voix

Tout le reste de l'assistant lit ce que la reconnaissance a écrit : une
phrase mal entendue est mal traduite, quoi que vaille le traducteur. Ce
notebook ne travaille que cet étage.

**Les données** : tout ce qui a un texte bambara exact, sans transcrire :
les **lectures** (onglet « Lire ») et les tours du mode **Discuter** dont le
bambara a été confirmé ou corrigé. Plus il y en a, plus l'affinage a de
chances de servir : 300 lectures, c'est un début ; quelques milliers, un
vrai corpus.

**Ce qui est garanti** : Soloni n'est remplacé que si la version affinée
1. fait **moins d'erreurs sur des lectures qu'elle n'a jamais entendues**
   (15 % des phrases lues, mises de côté) ;
2. et **comprend mieux tes phrases du jeu maison** (moitié test), avec le
   traducteur en service : c'est ce qui compte pour l'assistant.

Réciter ce qu'il a appris ne prouverait rien : il est jugé sur ce qu'il n'a
jamais entendu.

**Avant le grand entraînement**, `ESSAI = True` fait une seule époque pour
vérifier que tout s'enchaîne (quelques minutes, scores sans valeur).

*Exécution → Modifier le type d'exécution → GPU (T4)*, puis **Exécution →
Tout exécuter**. Compter 20 à 60 minutes selon le nombre d'enregistrements.

## Installation et Drive

In [1]:
# Branche de travail : contient les dernières corrections du notebook et des
# scripts. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

import os
os.chdir("/content")
if os.path.isdir("projet/.git"):
    !git -C projet pull -q   # relance : code mis à jour, rien à recloner
else:
    !git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd /content/projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

/content/projet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 14.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/bambara/modeles"
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"
ATELIER = "/content/drive/MyDrive/bambara/atelier"

Mounted at /content/drive


In [3]:
# Modèle final s'il existe, sinon le dernier checkpoint (entraînement coupé
# ou inachevé), sinon le modèle d'origine quand il y en a un. Défini ici pour
# rester disponible après une reprise (sections 1 à 3) ; les sections 6 à 10
# appellent chemins().
from pathlib import Path
from transformers.trainer_utils import get_last_checkpoint

def modele(dossier, repli=None):
    if (Path(dossier) / "model.safetensors").exists():
        return dossier
    dernier = get_last_checkpoint(dossier) if Path(dossier).is_dir() else None
    if dernier:
        print(f"Entraînement inachevé : {dernier}")
        return dernier
    if repli:
        print(f"Pas encore entraîné ({dossier}) : modèle d'origine {repli}")
        return repli
    raise FileNotFoundError(f"ni modèle final ni checkpoint dans {dossier}")

def deja(label):
    """Vrai si un rapport porte déjà ce nom : relancer une cellule
    d'évaluation, d'une session à l'autre, ne la refait pas. Sauf si l'un
    de ses modèles a été réentraîné depuis : l'ancien rapport est alors mis
    de côté (.ancien) et l'évaluation refaite."""
    import glob, json, os
    for p in glob.glob(f"{OUT}/*.json"):
        try:
            rapport = json.load(open(p))
        except ValueError:  # copie vers Drive interrompue : rapport tronqué
            os.rename(p, p + ".tronque")
            continue
        if rapport.get("nom") != label:
            continue
        poids = [f for m in rapport.get("modeles", {}).values() if isinstance(m, str)
                 for f in glob.glob(f"{m}/*.safetensors")]
        if any(os.path.getmtime(f) > os.path.getmtime(p) for f in poids):
            for ancien in (p, p[:-len(".json")] + ".details.jsonl"):
                if os.path.exists(ancien):
                    os.rename(ancien, ancien + ".ancien")
            print(f"« {label} » évalué avec un modèle réentraîné depuis : refait.")
            return False
        print(f"« {label} » déjà évalué ({p}) : cellule sautée.")
        return True
    return False

def a_jour(motif, code=False):
    """Vrai si la sortie la plus récente qui correspond au motif est
    postérieure aux poids de tous les modèles en service (chemins()) et,
    avec code=True, au dernier commit qui modifie la chaîne (bambara_voice) :
    sinon, l'étape est à refaire."""
    import glob, os, subprocess
    sorties = glob.glob(motif)
    if not sorties:
        return False
    modeles = {*chemins(), *meilleurs()}
    reperes = [os.path.getmtime(f) for m in modeles for f in glob.glob(f"{m}/*.safetensors")]
    if code:
        commit = subprocess.run(["git", "log", "-1", "--format=%ct", "--", "bambara_voice"],
                                capture_output=True, text=True).stdout.strip()
        reperes += [float(commit)] if commit else []
    derniere = max(os.path.getmtime(f) for f in sorties)
    return all(r < derniere for r in reperes)

def place_drive(besoin_go):
    """Arrête la cellule si Drive n'a pas la place : un Drive plein ne
    reçoit plus rien, sans aucune erreur dans Colab."""
    import shutil
    libre = shutil.disk_usage("/content/drive").free / 1e9
    print(f"Place libre sur Drive : {libre:.1f} Go (il en faut {besoin_go})")
    if libre < besoin_go:
        raise RuntimeError("Drive presque plein. Les fichiers supprimés restent dans la "
                           "corbeille et comptent encore : la vider sur drive.google.com.")

def synchroniser_drive(minutes=30):
    """Attend que Drive ait reçu tout ce qui a été écrit, puis le remonte
    (Colab peut redemander l'accès : accepter la fenêtre qui s'ouvre).
    Ne lève jamais d'erreur : « Tout exécuter » continue."""
    try:
        drive.flush_and_unmount(timeout_ms=minutes * 60 * 1000)
        print("Drive a tout reçu.")
    except Exception as e:
        print(f"Drive n'a pas tout reçu en {minutes} min ({e}). Drive plein ? Vider la "
              "corbeille sur drive.google.com, puis relancer cette cellule.")
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive non remonté ({e}) : relancer la section 2 avant la démo.")

def chemins():
    """Whisper, NLLB bambara → français et NLLB français → bambara à
    utiliser, d'après ce qui est sur Drive."""
    nllb = "facebook/nllb-200-distilled-600M"
    return (modele(f"{CKPT}/whisper-small-bm"),
            modele(f"{CKPT}/nllb-bm2fr", repli=nllb),
            modele(f"{CKPT}/nllb-fr2bm", repli=nllb))

RETENU = f"{CKPT}/retenu.json"           # l'assemblage en service et ses preuves
MOYENNE = "/content/nllb-bm2fr-moyenne"  # moyenne des poids v1/v2 (§9a), refaite au besoin

def en_service():
    """L'assemblage en service (eval.promotion) : Whisper + NLLB v1, celui de
    la démo du 15 octobre, tant qu'aucun candidat n'a fait mieux, mesure à
    l'appui, sur le test maison. Un nouveau modèle ne le remplace que là."""
    from eval.promotion import lire_retenu
    whisper, bm2fr, fr2bm = chemins()
    r = lire_retenu(RETENU, {"asr": {"model_id": whisper}, "nom_asr": "Whisper",
                             "bm2fr": bm2fr, "nom_bm2fr": "NLLB v1",
                             "fr2bm": fr2bm, "nom_fr2bm": "NLLB v1"})
    if r["bm2fr"] == MOYENNE and not Path(f"{MOYENNE}/model.safetensors").exists():
        import subprocess  # la moyenne vit sur le disque de Colab : une minute
        subprocess.run(["python", "scripts/moyenne_poids.py", "--a", bm2fr,
                        "--b", f"{CKPT}/nllb-bm2fr-v2", "--out", MOYENNE], check=True)
    return r

def meilleurs():
    """Les modèles de l'assistant (§9 à 11) : Whisper (v2 si entraîné) et les
    deux NLLB en service."""
    r = en_service()
    whisper = chemins()[0]
    if Path(f"{CKPT}/whisper-small-bm-v2/model.safetensors").exists():
        whisper = f"{CKPT}/whisper-small-bm-v2"
    return whisper, r["bm2fr"], r["fr2bm"]

## Ce qui est en service, mesuré sur ta voix

In [4]:
import glob, json, os
from eval.promotion import score
WHISPER, NLLB_BM2FR, NLLB_FR2BM = meilleurs()
EN_SERVICE = en_service()
CFG = "/content/assistant-affine.json"
import torch  # sans GPU, sur processeur : plus lent, mais sans quota
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
json.dump({"device": DEVICE, "asr": EN_SERVICE["asr"], "mt_in": {"model_id": NLLB_BM2FR},
           "mt_out": {"model_id": NLLB_FR2BM}}, open(CFG, "w"), indent=1)
MAISON, PRISES = f"{ATELIER}/testset-dire.jsonl", f"{ATELIER}/dire.jsonl"
TITULAIRE = f"{EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']}"
RAPPORT_EN_SERVICE = None
if not os.path.exists(PRISES):
    print("Aucune phrase enregistrée avec l'atelier : la v4 ne peut pas être jugée.")
else:
    if EN_SERVICE["asr"].get("kind") == "nemo":   # Soloni : jugement de la v4 au §5
        !pip install -q "nemo_toolkit[asr]"
    !python -m eval.atelier --racine "$ATELIER" exporter > /dev/null
    dossier = f"{OUT}/maison-assemblages/{EN_SERVICE['nom_asr']}-{EN_SERVICE['nom_bm2fr']}".replace(" ", "-")
    motif = f"{dossier}/cascade-*[0-9].json"
    if not (glob.glob(motif) and max(map(os.path.getmtime, glob.glob(motif))) > os.path.getmtime(PRISES)):
        print(f"{TITULAIRE} : mesure sur tes phrases…")
        !python -m eval.run_eval --testset "$MAISON" --arch cascade --config $CFG \
            --comprendre-seulement --nom "$TITULAIRE" --out "$dossier" > /dev/null 2>&1
    if glob.glob(motif):
        RAPPORT_EN_SERVICE = sorted(glob.glob(motif))[-1]
        print(f"En service : {TITULAIRE} ; chrF++ sur ta voix : dev "
              f"{score(RAPPORT_EN_SERVICE, 'dev'):.1f}, test {score(RAPPORT_EN_SERVICE, 'test'):.1f}")
    else:
        print(f"⚠️ {TITULAIRE} n'a pas pu être mesuré : relancer la cellule.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 26.6 MB/s eta 0:00:00
   ━━━

## Réglages

In [5]:
ESSAI = False   # True : une époque, pour vérifier que tout s'enchaîne
EPOCHS = 30     # au plus : l'entraînement s'arrête quand Soloni ne progresse plus

## Affiner Soloni

Toujours à partir du Soloni de RobotsMali, avec tous les enregistrements au
texte exact : chaque version s'appelle « perso-N », N étant leur nombre.
Après de nouvelles lectures ou discussions, relancer : N augmente, une
nouvelle version est entraînée et jugée.

In [6]:
import json, os, shutil, sys
from pathlib import Path
sys.path.insert(0, "scripts")
from affiner_soloni import SOLONI, exemples
N_VOIX = len(exemples(Path(ATELIER)))
VERSION_VOIX = f"perso-{N_VOIX}"
DOSSIER_VOIX = f"{CKPT}/soloni-{VERSION_VOIX}"          # sur Drive : modèle et rapport
SOLONI_VOIX = f"{DOSSIER_VOIX}/soloni-perso.nemo"
LOCAL_VOIX = "/content/soloni-essai" if ESSAI else f"/content/soloni-{VERSION_VOIX}"
print(f"{N_VOIX} enregistrements au texte exact (lectures et tours corrigés).")
if Path(SOLONI_VOIX).exists() and not ESSAI:
    print(f"Soloni {VERSION_VOIX} déjà affiné.")
elif N_VOIX < 30:
    print("Trop peu d'enregistrements pour affiner : enregistre des lectures, ou discute "
          "avec l'assistant (mode entraînement), puis relance.")
else:
    ARG_ESSAI = "--essai" if ESSAI else ""
    !python scripts/affiner_soloni.py --atelier "$ATELIER" --out "$LOCAL_VOIX" \
        --modele "$SOLONI" --epochs $EPOCHS $ARG_ESSAI
    if not ESSAI and Path(f"{LOCAL_VOIX}/soloni-perso.nemo").exists():
        place_drive(1)
        os.makedirs(DOSSIER_VOIX, exist_ok=True)
        for f in ("soloni-perso.nemo", "rapport.json"):
            shutil.copy2(f"{LOCAL_VOIX}/{f}", DOSSIER_VOIX)
        print(f"Enregistré dans {DOSSIER_VOIX}")

1 enregistrements au texte exact (lectures et tours corrigés).
Trop peu d'enregistrements pour affiner : enregistre des lectures, ou discute avec l'assistant (mode entraînement), puis relance.


## Le jugement

Les deux conditions : moins d'erreurs sur les lectures de test, et une
meilleure compréhension de tes phrases (moitié test du jeu maison, IC à
95 % au-dessus de zéro), avec le traducteur en service. La décision va dans
`retenu.json` ; la démo et l'atelier la lisent.

In [7]:
import glob, json, os
from pathlib import Path
from eval.promotion import comparer, ecrire_retenu, to_markdown, verdict
if ESSAI:
    print("Essai : rien à juger. Remettre ESSAI = False pour le vrai entraînement.")
elif not RAPPORT_EN_SERVICE:
    print("Pas de mesure de tes phrases : impossible de juger.")
elif not Path(SOLONI_VOIX).exists():
    print("Pas de Soloni affiné à juger.")
elif EN_SERVICE["asr"].get("model_id") == SOLONI_VOIX:
    print(f"Soloni {VERSION_VOIX} est déjà en service.")
else:
    lectures = json.load(open(f"{DOSSIER_VOIX}/rapport.json"))
    avant, apres = lectures["wer_test_avant"], lectures["wer_test_apres"]
    nom_voix = f"Soloni {VERSION_VOIX} + {EN_SERVICE['nom_bm2fr']}"
    dossier = f"{OUT}/maison-assemblages/{nom_voix}".replace(" + ", "-").replace(" ", "-")
    cfg_voix = "/content/assemblage-voix.json"
    json.dump({"device": DEVICE, "asr": {"model_id": SOLONI_VOIX, "kind": "nemo"},
               "mt_in": {"model_id": EN_SERVICE["bm2fr"]}, "mt_out": {"model_id": EN_SERVICE["fr2bm"]}},
              open(cfg_voix, "w"), indent=1)
    motif = f"{dossier}/cascade-*[0-9].json"
    if not (glob.glob(motif) and max(map(os.path.getmtime, glob.glob(motif))) > os.path.getmtime(PRISES)):
        print(f"{nom_voix} : mesure sur tes phrases…")
        !python -m eval.run_eval --testset "$MAISON" --arch cascade --config $cfg_voix \
            --comprendre-seulement --nom "$nom_voix" --out "$dossier" > /dev/null 2>&1
    if not glob.glob(motif):
        print("⚠️ La mesure sur tes phrases a échoué : relancer la cellule.")
    else:
        rapport_voix = sorted(glob.glob(motif))[-1]
        decision = verdict(comparer(rapport_voix, RAPPORT_EN_SERVICE, "test"))
        decision["wer_lectures"] = {"avant": avant, "apres": apres}
        print(f"Lectures jamais entendues ({avant['n']}) : WER {avant['wer']} % → {apres['wer']} % "
              f"(relâché {avant['wer_releche']} % → {apres['wer_releche']} %)")
        if apres["wer"] >= avant["wer"]:
            decision["promu"] = False
            decision["raisons"].append(f"pas moins d'erreurs sur les lectures de test : "
                                       f"{avant['wer']} % → {apres['wer']} %")
        print(to_markdown(decision, nom_voix, TITULAIRE))
        EN_SERVICE = ecrire_retenu(RETENU, EN_SERVICE, "reconnaissance (voix)",
                                   {"asr": {"model_id": SOLONI_VOIX, "kind": "nemo"},
                                    "nom_asr": f"Soloni {VERSION_VOIX}"}, decision)
        if decision["promu"]:
            RAPPORT_EN_SERVICE = rapport_voix
    print(f"En service : {EN_SERVICE['nom_asr']} + {EN_SERVICE['nom_bm2fr']}")

Pas de Soloni affiné à juger.
